### Importing libraries:

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report, accuracy_score,
    roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

### data understanding, loading and cleaning:

In [60]:

def load_and_clean_data(
    hist_path: str | Path = r"C:\Users\anil\Documents\JOB assignments\Primetrade.ai\historical_data.csv",
    fgi_path:  str | Path = r"C:\Users\anil\Documents\JOB assignments\Primetrade.ai\fear_greed_index.csv",
) -> pd.DataFrame:
    hist_path, fgi_path = Path(hist_path), Path(fgi_path)

    # ── Load ─────────────────────────────────────────────────────────────────
    if not hist_path.exists():
        raise FileNotFoundError(f"Historical data not found: {hist_path}")
    if not fgi_path.exists():
        raise FileNotFoundError(f"FGI data not found: {fgi_path}")

    hist_df = pd.read_csv(hist_path)
    fgi_df  = pd.read_csv(fgi_path)

    # ── Quality report ───────────────────────────────────────────────────────
    print(f"[hist] rows={len(hist_df):,}  missing={hist_df.isnull().sum().sum():,}  "
          f"duplicates={hist_df.duplicated().sum():,}")
    print(f"[fgi]  rows={len(fgi_df):,}  missing={fgi_df.isnull().sum().sum():,}  "
          f"duplicates={fgi_df.duplicated().sum():,}")

    # ── Drop duplicates (don't just report them) ─────────────────────────────
    hist_df = hist_df.drop_duplicates()
    fgi_df  = fgi_df.drop_duplicates()

    # ── Parse & validate historical timestamps ───────────────────────────────
    hist_df['Datetime'] = pd.to_datetime(
        hist_df['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce'
    )
    bad_ts = hist_df['Datetime'].isna().sum()
    if bad_ts:
        print(f"[warn] Dropping {bad_ts:,} hist rows with unparseable timestamps.")
        hist_df = hist_df.dropna(subset=['Datetime'])

    hist_df['Date'] = hist_df['Datetime'].dt.normalize()   # midnight-aligned Timestamp

    # ── Parse & validate FGI dates ───────────────────────────────────────────
    fgi_df['Date'] = pd.to_datetime(fgi_df['date'], errors='coerce')
    bad_dates = fgi_df['Date'].isna().sum()
    if bad_dates:
        print(f"[warn] Dropping {bad_dates:,} FGI rows with unparseable dates.")
        fgi_df = fgi_df.dropna(subset=['Date'])

    # ── Validate FGI 'value' is numeric ──────────────────────────────────────
    fgi_df['value'] = pd.to_numeric(fgi_df['value'], errors='coerce')
    if fgi_df['value'].isna().any():
        print(f"[warn] {fgi_df['value'].isna().sum()} non-numeric FGI values → NaN.")

    # ── Sentiment simplification ──────────────────────────────────────────────
    def simplify_fgi(c: str) -> str:
        c = str(c).lower()
        if 'fear'  in c: return 'Fear'
        if 'greed' in c: return 'Greed'
        return 'Neutral'

    fgi_df['Sentiment'] = fgi_df['classification'].apply(simplify_fgi)

    # ── FIX: date-offset lag instead of row-index .shift(1) ──────────────────
    # .shift(1) moves up one *row*, which is wrong whenever the FGI series has
    # gaps (weekends, holidays).  The safe approach: build a lookup keyed on
    # (today's date + 1 day), so the merge naturally pairs each trading date
    # with the FGI value that was *published the day before*.
    fgi_lookup = (
        fgi_df[['Date', 'value', 'Sentiment']]
        .drop_duplicates('Date')
        .sort_values('Date')
        .copy()
    )
    fgi_lookup['Lookup_Date'] = fgi_lookup['Date'] + pd.DateOffset(days=1)
    fgi_lookup = fgi_lookup.rename(columns={
        'value':     'FGI_Value',
        'Sentiment': 'Sentiment',
    })[['Lookup_Date', 'FGI_Value', 'Sentiment']]

    # ── Merge ────────────────────────────────────────────────────────────────
    before = len(hist_df)
    df = pd.merge(
        hist_df,
        fgi_lookup.rename(columns={'Lookup_Date': 'Date'}),
        on='Date',
        how='inner',
    )
    after = len(df)
    dropped = before - after
    if dropped:
        print(f"[info] Merge dropped {dropped:,} hist rows ({dropped/before:.1%}) "
              f"with no matching FGI entry.")
    if after == 0:
        raise ValueError("Merge produced an empty DataFrame. Check date formats / ranges.")

    print(f"[info] Merged dataset: {after:,} rows spanning "
          f"{df['Date'].min().date()} → {df['Date'].max().date()}")

    return df

### Feature engineering:

In [61]:
def feature_engineering(df):
    print("Engineering key metrics...")
    daily_rows = []
 
    for (account, date, sentiment), group in df.groupby(['Account', 'Date', 'Sentiment']):
        daily_pnl       = group['Closed PnL'].sum()
        total_trades    = len(group)
        avg_trade_size  = group['Size USD'].mean()
 
        closing_trades = group[group['Closed PnL'] != 0]
        win_rate = (
            (closing_trades['Closed PnL'] > 0).sum() / len(closing_trades)
            if len(closing_trades) > 0 else np.nan
        )
 
        long_trades  = (group['Side'].str.upper() == 'BUY').sum()
        short_trades = (group['Side'].str.upper() == 'SELL').sum()
        ls_ratio     = long_trades / (short_trades + 1e-5)
 
        fgi_value = group['FGI_Value'].iloc[0] if 'FGI_Value' in group.columns else np.nan
 
        daily_rows.append({
            'Account':           account,
            'Date':              date,
            'Sentiment':         sentiment,
            'FGI_Value':         fgi_value,
            'Daily_PnL':         daily_pnl,
            'Total_Trades':      total_trades,
            'Avg_Trade_Size_USD': avg_trade_size,
            'Win_Rate':          win_rate,
            'Long_Short_Ratio':  ls_ratio,
        })
 
    daily_df = pd.DataFrame(daily_rows).sort_values(['Account', 'Date'])
 
    # FIX: add properly lagged features so the model learns *history*, not the
    #      current day's outcome (which is tightly coupled to the target).
    daily_df = daily_df.sort_values(['Account', 'Date'])
    for col in ['Daily_PnL', 'Win_Rate', 'Total_Trades', 'Long_Short_Ratio']:
        daily_df[f'{col}_lag1'] = daily_df.groupby('Account')[col].shift(1)
        daily_df[f'{col}_roll3'] = (
            daily_df.groupby('Account')[col]
            .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        )
 
    # Cumulative PnL streak — captures momentum without using same-day values
    daily_df['PnL_Streak'] = (
        daily_df.groupby('Account')['Daily_PnL']
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    )
 
    median_size = daily_df['Avg_Trade_Size_USD'].median()
    daily_df['Volume_Segment'] = np.where(
        daily_df['Avg_Trade_Size_USD'] > median_size, 'High_Volume', 'Low_Volume'
    )
 
    return daily_df
 

### Charts :

In [62]:
def generate_charts(daily_df):
    print("Generating charts...")
    sns.set_theme(style="whitegrid")
    df_filtered = daily_df[daily_df['Sentiment'].isin(['Fear', 'Greed'])]
 
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
    sns.barplot(data=df_filtered, x='Sentiment', y='Daily_PnL',
                estimator=np.mean, ci=None, palette='muted', ax=axes[0])
    axes[0].set_title("Avg Daily PnL by Sentiment")
    axes[0].set_ylabel("Avg Daily PnL (USD)")
 
    sns.barplot(data=df_filtered, x='Sentiment', y='Total_Trades',
                estimator=np.mean, ci=None, palette='pastel', ax=axes[1])
    axes[1].set_title("Avg Trade Frequency by Sentiment")
    axes[1].set_ylabel("Trades per Day")
 
    sns.barplot(data=df_filtered, x='Volume_Segment', y='Daily_PnL',
                hue='Sentiment', ci=None, ax=axes[2])
    axes[2].set_title("Performance: High vs Low Volume Segments")
    axes[2].set_ylabel("Avg Daily PnL (USD)")
 
    plt.tight_layout()
    plt.savefig("sentiment_analysis.png", dpi=150)
    plt.close()
    print("Saved sentiment_analysis.png")
 

### Model building and evaluation:

In [63]:
def build_predictive_model(df):
    print("\nTraining next-day profitability model...")
 
    df = df.sort_values(['Account', 'Date']).copy()
 
    # Target: is tomorrow profitable?
    df['Next_Day_PnL']          = df.groupby('Account')['Daily_PnL'].shift(-1)
    df['Is_Profitable_Next_Day'] = (df['Next_Day_PnL'] > 0).astype(int)
 
    # FIX: drop same-day raw PnL/Win_Rate — use only lagged versions to avoid
    #      leakage (current-day outcome → next-day target shortcut).
    FEATURES = [
        'Daily_PnL_lag1',    'Daily_PnL_roll3',   'PnL_Streak',
        'Win_Rate_lag1',     'Win_Rate_roll3',
        'Total_Trades_lag1', 'Total_Trades_roll3',
        'Long_Short_Ratio_lag1',
        'Avg_Trade_Size_USD',
        'FGI_Value',
        # One-hot sentiment (fitted on train only — done inside the split loop)
    ]
 
    model_df = df.dropna(subset=['Next_Day_PnL']).copy()
    model_df['Sentiment_Fear']  = (model_df['Sentiment'] == 'Fear').astype(int)
    model_df['Sentiment_Greed'] = (model_df['Sentiment'] == 'Greed').astype(int)
    FEATURES += ['Sentiment_Fear', 'Sentiment_Greed']
 
    model_df = model_df.dropna(subset=[f for f in FEATURES if 'lag1' in f or 'roll3' in f])
 
    X = model_df[FEATURES].copy()
    y = model_df['Is_Profitable_Next_Day']
 
    # ── Class balance check ──────────────────────────────────────────────────
    pos_rate = y.mean()
    print(f"Class balance — Profitable: {pos_rate:.1%} | Not profitable: {1-pos_rate:.1%}")
    class_weight = 'balanced' if not (0.4 < pos_rate < 0.6) else None
    if class_weight:
        print("Imbalanced classes detected — using class_weight='balanced'")
 
    # FIX: time-based split — never shuffle time-series data.
    #      Use the last 20 % of chronological rows as the test set.
    split_idx = int(len(model_df) * 0.80)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
 
    # FIX: imputation fitted on train only, then applied to test.
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),   # fit on train only
        ('scaler',  StandardScaler()),                   # not needed for RF but
        ('model',   RandomForestClassifier(             #   keeps pipeline reusable
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=10,   # regularise against overfitting
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1,
        )),
    ])
 
    # ── Time-series cross-validation on training fold ────────────────────────
    print("\nTime-series cross-validation (5 folds) on training data:")
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), 1):
        pipeline.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        val_preds = pipeline.predict(X_train.iloc[val_idx])
        score = accuracy_score(y_train.iloc[val_idx], val_preds)
        cv_scores.append(score)
        print(f"  Fold {fold}: {score:.3f}")
    print(f"  Mean CV Accuracy: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}")
 
    # ── Final fit on full training set, evaluate on held-out test ───────────
    pipeline.fit(X_train, y_train)
    preds      = pipeline.predict(X_test)
    proba      = pipeline.predict_proba(X_test)[:, 1]
 
    print(f"\nHeld-out Test Accuracy : {accuracy_score(y_test, preds):.3f}")
    print(f"Held-out ROC-AUC       : {roc_auc_score(y_test, proba):.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=['Not Profitable', 'Profitable']))
 
    # ── Feature importance ───────────────────────────────────────────────────
    rf_model     = pipeline.named_steps['model']
    importances  = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("\nTop-10 Feature Importances:")
    print(importances.head(10).to_string())
 
    fig, ax = plt.subplots(figsize=(8, 5))
    importances.head(10).plot(kind='barh', ax=ax)
    ax.invert_yaxis()
    ax.set_title("Random Forest — Top-10 Feature Importances")
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150)
    plt.close()
 
    # ── Confusion matrix ─────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds,
        display_labels=['Not Profitable', 'Profitable'],
        ax=ax, colorbar=False
    )
    plt.title("Confusion Matrix (held-out test)")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.close()
 
    joblib.dump(pipeline, 'profit_predictor.pkl')
    print("\nPipeline saved as profit_predictor.pkl")
 

### Loading Pipeline to pickle:

In [64]:
if __name__ == "__main__":
    # ── Paths ─────────────────────────────────────────────────────────────────
    BASE_DIR  = Path(r"C:\Users\anil\Documents\JOB assignments\Primetrade.ai")
    hist_path = BASE_DIR / "historical_data.csv"
    fgi_path  = BASE_DIR / "fear_greed_index.csv"
    out_csv   = BASE_DIR / "processed_daily_metrics.csv"

    # ── Run pipeline ──────────────────────────────────────────────────────────
    merged_df     = load_and_clean_data(hist_path, fgi_path)
    daily_metrics = feature_engineering(merged_df)
    daily_metrics.to_csv(out_csv, index=False)
    print(f"[info] Metrics saved → {out_csv}")
    generate_charts(daily_metrics)
    build_predictive_model(daily_metrics)
    print("\nPipeline complete. Assets saved.")

[hist] rows=211,224  missing=0  duplicates=0
[fgi]  rows=2,644  missing=0  duplicates=0
[info] Merge dropped 4 hist rows (0.0%) with no matching FGI entry.
[info] Merged dataset: 211,220 rows spanning 2023-05-01 → 2025-05-01
Engineering key metrics...
[info] Metrics saved → C:\Users\anil\Documents\JOB assignments\Primetrade.ai\processed_daily_metrics.csv
Generating charts...
Saved sentiment_analysis.png

Training next-day profitability model...
Class balance — Profitable: 70.9% | Not profitable: 29.1%
Imbalanced classes detected — using class_weight='balanced'
Train: 1312 rows | Test: 328 rows

Time-series cross-validation (5 folds) on training data:
  Fold 1: 0.725
  Fold 2: 0.486
  Fold 3: 0.541
  Fold 4: 0.683
  Fold 5: 0.592
  Mean CV Accuracy: 0.606 ± 0.088

Held-out Test Accuracy : 0.573
Held-out ROC-AUC       : 0.603

Classification Report:
                precision    recall  f1-score   support

Not Profitable       0.39      0.46      0.42       110
    Profitable       0.70  